# Nykaa Domain Support Agent — Task 12: Structured Logging

## Objective
Task 12 requires **every HTTP request** to produce exactly one JSON-Lines (`.jsonl`) log entry containing:
- a unique trace ID;
- request timing information;
- request method, path and status code;
- request text with the same fixed-format PII masking used by the Task 10 guardrail.

The middleware created here wraps your existing `app.py`; it does not replace your RAG or order-lookup logic. It writes a separate file, `task12_structured_logs.jsonl`.

**Important:** Run this notebook from the folder containing your existing `app.py`, `orders.csv` (or `orders_with_escalation_scores.csv`) and `chroma_db/`.

## 1. Check the existing FastAPI project

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
APP_FILE = PROJECT_DIR / "app.py"
CHROMA_DIR = PROJECT_DIR / "chroma_db"
LOG_FILE = PROJECT_DIR / "task12_structured_logs.jsonl"

print("Project directory:", PROJECT_DIR.resolve())
print("FastAPI app:", APP_FILE.exists())
print("ChromaDB folder:", CHROMA_DIR.exists())

if not APP_FILE.exists():
    raise FileNotFoundError("app.py not found. Run this notebook from your Task 11 project folder.")
if not CHROMA_DIR.exists():
    raise FileNotFoundError("chroma_db folder not found.")


Project directory: C:\Users\singh\Downloads
FastAPI app: True
ChromaDB folder: True


## 2. Create the structured-logging middleware

This creates `observability_app.py`, which imports your existing FastAPI application and adds one ASGI middleware layer.

The middleware buffers the incoming request body, creates a **masked copy for logging**, and then passes the original body to the existing application. Your Task 10 application guardrail remains responsible for masking the query before retrieval. The middleware never writes the raw request body to the Task 12 log.

Each HTTP request receives one JSONL entry, including health checks, policy/order requests, document ingestion and validation errors.

In [2]:
%%writefile observability_app.py
import json
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from threading import Lock

from app import app as fastapi_app

BASE_DIR = Path(__file__).resolve().parent
STRUCTURED_LOG_PATH = BASE_DIR / "task12_structured_logs.jsonl"
_log_lock = Lock()

# Keep these patterns aligned with the fixed-format PII patterns in Task 10.
PHONE_PATTERN = re.compile(
    r"(?<!\d)(?:\+?91[\s-]?)?[6-9]\d{2}[\s-]?\d{3}[\s-]?\d{4}(?!\d)"
)
CARD_LAST4_PATTERN = re.compile(
    r"(?i)(\b(?:card|credit\s*card|debit\s*card|payment\s*card)"
    r"\s*(?:last\s*4|last\s*four|ending\s*in|ending|last4)"
    r"\s*(?:(?:is|:|#|-)\s*)?)(\d{4})\b"
)

def mask_text_for_log(text: str) -> str:
    text = PHONE_PATTERN.sub("[PHONE_MASKED]", text)
    text = CARD_LAST4_PATTERN.sub(
        lambda match: match.group(1) + "[CARD_LAST4_MASKED]", text
    )
    return text

def append_jsonl(record: dict) -> None:
    line = json.dumps(record, ensure_ascii=False)
    with _log_lock:
        with STRUCTURED_LOG_PATH.open("a", encoding="utf-8") as file:
            file.write(line + "\n")

class StructuredLoggingMiddleware:
    def __init__(self, app):
        self.app = app

    async def __call__(self, scope, receive, send):
        if scope["type"] != "http":
            await self.app(scope, receive, send)
            return

        started = time.perf_counter()
        trace_id = str(uuid.uuid4())
        method = scope.get("method", "UNKNOWN")
        path = scope.get("path", "/")
        status_code = 500
        request_chunks = []

        # Read the request body once so a safe copy can be logged.
        more_body = True
        while more_body:
            message = await receive()
            if message["type"] == "http.disconnect":
                break
            if message["type"] == "http.request":
                request_chunks.append(message.get("body", b""))
                more_body = message.get("more_body", False)

        raw_body = b"".join(request_chunks)
        body_text = raw_body.decode("utf-8", errors="replace")
        masked_body = mask_text_for_log(body_text)

        # Replay the original body unchanged to FastAPI.
        body_sent = False
        async def replay_receive():
            nonlocal body_sent
            if not body_sent:
                body_sent = True
                return {
                    "type": "http.request",
                    "body": raw_body,
                    "more_body": False
                }
            return {
                "type": "http.request",
                "body": b"",
                "more_body": False
            }

        async def send_with_trace(message):
            nonlocal status_code
            if message["type"] == "http.response.start":
                status_code = message["status"]
                headers = list(message.get("headers", []))
                headers.append((b"x-trace-id", trace_id.encode("ascii")))
                message = {**message, "headers": headers}
            await send(message)

        try:
            await self.app(scope, replay_receive, send_with_trace)
        finally:
            duration_ms = round((time.perf_counter() - started) * 1000, 3)
            record = {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "trace_id": trace_id,
                "method": method,
                "path": path,
                "status_code": status_code,
                "duration_ms": duration_ms,
                "request_text": masked_body[:10000]
            }
            append_jsonl(record)

fastapi_app.add_middleware(StructuredLoggingMiddleware)
app = fastapi_app


Writing observability_app.py


## 3. Import the instrumented application

Use `observability_app:app` when starting Uvicorn for Task 12. This keeps the original `app.py` intact and adds the logging middleware around it.

In [3]:
import observability_app

instrumented_app = observability_app.app

print("Instrumented API title:", instrumented_app.title)
print("Structured log path:", observability_app.STRUCTURED_LOG_PATH.resolve())
print("Middleware installed:", any(
    middleware.cls.__name__ == "StructuredLoggingMiddleware"
    for middleware in instrumented_app.user_middleware
))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Instrumented API title: Nykaa Domain Support Agent API
Structured log path: C:\Users\singh\Downloads\task12_structured_logs.jsonl
Middleware installed: True


## 4. Test that every request creates exactly one log entry

The tests below issue requests to health, policy, order and document-ingestion endpoints. They also test a validation-error request. The log count should increase by exactly one for each HTTP request.

In [4]:
from fastapi.testclient import TestClient
import json

task12_client = TestClient(instrumented_app)
task12_log = observability_app.STRUCTURED_LOG_PATH

# Start with a clean Task 12 evidence log so the request count is easy to verify.
if task12_log.exists():
    task12_log.unlink()

responses = []

responses.append(task12_client.get("/health"))

responses.append(task12_client.post("/ask", json={
    "conversation_id": "task12_policy_test",
    "query": "How long does a cash-on-delivery refund take?"
}))

# Use an order ID from the actual synthetic dataset.
import app as existing_app
sample_order_id = str(existing_app.orders_df.iloc[0]["record_id"])

responses.append(task12_client.post("/ask", json={
    "conversation_id": "task12_order_test",
    "query": f"Please check order {sample_order_id}."
}))

responses.append(task12_client.post("/add-document", json={
    "document_id": "task12_logging_test_policy",
    "source": "task12_logging_test_policy.md",
    "text": (
        "This synthetic policy is used to test the document-ingestion endpoint. "
        "It is not an official Nykaa policy."
    )
}))

# Deliberately invalid body: FastAPI should return 422, and middleware should still log it.
responses.append(task12_client.post("/ask", json={
    "conversation_id": "task12_invalid_test"
}))

print("HTTP status codes:", [response.status_code for response in responses])
assert [r.status_code for r in responses] == [200, 200, 200, 200, 422]

entries = [
    json.loads(line)
    for line in task12_log.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

print("HTTP requests sent:", len(responses))
print("JSONL entries written:", len(entries))

assert len(entries) == len(responses), "Expected exactly one log entry per request."
assert all(entry["trace_id"] for entry in entries)
assert all(entry["duration_ms"] >= 0 for entry in entries)
assert all("status_code" in entry for entry in entries)

print("PASS: Exactly one structured JSONL entry was written per HTTP request.")


HTTP status codes: [200, 200, 200, 200, 422]
HTTP requests sent: 5
JSONL entries written: 5
PASS: Exactly one structured JSONL entry was written per HTTP request.


## 5. Test PII masking in the log

Send fabricated phone and card-last-four values through `/ask`. The application receives the original request and applies its Task 10 guardrail; the Task 12 middleware must write only masked values to disk.

In [5]:
pii_test_response = task12_client.post("/ask", json={
    "conversation_id": "task12_pii_test",
    "query": (
        "My phone is 9876543210 and my payment card last 4 is 4321. "
        "How long does a COD refund take?"
    )
})

print("HTTP status:", pii_test_response.status_code)
print("Response query:", pii_test_response.json().get("query"))

log_contents = task12_log.read_text(encoding="utf-8")

assert pii_test_response.status_code == 200
assert "9876543210" not in log_contents
assert "4321" not in log_contents
assert "[PHONE_MASKED]" in log_contents
assert "[CARD_LAST4_MASKED]" in log_contents

print("PASS: Raw phone and card-last-four values are absent from the Task 12 log.")


HTTP status: 200
Response query: My phone is [PHONE_MASKED] and my payment card last 4 is [CARD_LAST4_MASKED]. How long does a COD refund take?
PASS: Raw phone and card-last-four values are absent from the Task 12 log.


## 6. Validate the JSONL structure and timing fields

Every non-empty line must be independently parseable JSON. Check required fields, unique trace IDs and numeric timing values.

In [6]:
all_lines = [
    line for line in task12_log.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
all_entries = [json.loads(line) for line in all_lines]

required_log_fields = {
    "timestamp_utc", "trace_id", "method", "path",
    "status_code", "duration_ms", "request_text"
}

for entry in all_entries:
    assert required_log_fields.issubset(entry.keys())
    assert isinstance(entry["duration_ms"], (int, float))
    assert entry["duration_ms"] >= 0
    assert isinstance(entry["status_code"], int)

trace_ids = [entry["trace_id"] for entry in all_entries]
assert len(trace_ids) == len(set(trace_ids)), "Trace IDs should be unique."

print("Total log entries:", len(all_entries))
print("Unique trace IDs:", len(set(trace_ids)))
print("Status-code counts:")
from collections import Counter
print(dict(Counter(entry["status_code"] for entry in all_entries)))

print("\nLast 3 structured log entries:")
for entry in all_entries[-3:]:
    print(json.dumps(entry, indent=2, ensure_ascii=False))

print("\nPASS: JSONL schema, trace IDs and timing fields validated.")


Total log entries: 6
Unique trace IDs: 6
Status-code counts:
{200: 5, 422: 1}

Last 3 structured log entries:
{
  "timestamp_utc": "2026-11-04T20:37:39.972632+00:00",
  "trace_id": "b5d02af9-f974-4052-8375-e4cbcf099e0d",
  "method": "POST",
  "path": "/add-document",
  "status_code": 200,
  "duration_ms": 36.918,
  "request_text": "{\"document_id\":\"task12_logging_test_policy\",\"source\":\"task12_logging_test_policy.md\",\"text\":\"This synthetic policy is used to test the document-ingestion endpoint. It is not an official Nykaa policy.\"}"
}
{
  "timestamp_utc": "2026-11-04T20:37:39.979126+00:00",
  "trace_id": "44c795d2-4095-4e08-a6c9-2b97c7de95dd",
  "method": "POST",
  "path": "/ask",
  "status_code": 422,
  "duration_ms": 1.829,
  "request_text": "{\"conversation_id\":\"task12_invalid_test\"}"
}
{
  "timestamp_utc": "2026-11-04T20:37:40.066795+00:00",
  "trace_id": "333a840d-a0f8-4736-8142-1d931d1cb25c",
  "method": "POST",
  "path": "/ask",
  "status_code": 200,
  "duration_ms"

## 7. Run the instrumented API in VS Code

Stop the old Uvicorn process with `Ctrl+C`. In the VS Code terminal, make sure you are in the project directory and run:

```powershell
python -m uvicorn observability_app:app --host 127.0.0.1 --port 8000 --reload
```

Open `http://127.0.0.1:8000/docs` and make a few requests. Each request should append one entry to:

```text
task12_structured_logs.jsonl
```

The response headers should also include `x-trace-id`. The Task 12 notebook tests use `TestClient`, so they can validate the middleware without running a separate terminal server.

## 8. Task 12 completion checklist

Task 12 is complete when:
- every HTTP request handled by the instrumented FastAPI app creates exactly one JSONL entry;
- each entry contains a unique trace ID and timing information;
- the log contains method, path, response status and request text;
- raw fixed-format phone/card-last-four PII values are absent from the log;
- validation-error requests are logged too.

**Evidence file:** `task12_structured_logs.jsonl`

The original brief's next task is Task 13: evaluate 15 queries with the RAG triad under `MOCK_LLM`, report all three scores per query, and calculate the three averages.

In [7]:
import json
from pathlib import Path

log_file = Path("task12_structured_logs.jsonl")

assert log_file.exists(), "Log file not found!"

with open(log_file, "r", encoding="utf-8") as f:
    entries = [json.loads(line) for line in f if line.strip()]

print("TOTAL LOG ENTRIES:", len(entries))

# Check trace IDs
assert all(e["trace_id"] for e in entries)
print("PASS: Trace IDs generated.")

# Check timing
assert all(e["duration_ms"] >= 0 for e in entries)
print("PASS: Request timing recorded.")

# Check HTTP metadata
assert all(
    e["method"] and e["path"] and e["status_code"]
    for e in entries
)
print("PASS: HTTP metadata recorded.")

# Check PII masking
log_text = log_file.read_text(encoding="utf-8")

assert "9876543210" not in log_text
assert "4321" not in log_text

assert "[PHONE_MASKED]" in log_text
assert "[CARD_LAST4_MASKED]" in log_text

print("PASS: PII masking verified.")

# Check unique trace IDs
trace_ids = [e["trace_id"] for e in entries]
assert len(trace_ids) == len(set(trace_ids))

print("PASS: Trace IDs are unique.")

print("\\nTASK 12 VALIDATION COMPLETED.")

TOTAL LOG ENTRIES: 12
PASS: Trace IDs generated.
PASS: Request timing recorded.
PASS: HTTP metadata recorded.
PASS: PII masking verified.
PASS: Trace IDs are unique.
\nTASK 12 VALIDATION COMPLETED.
